[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-05-01-einsum/challenge.ipynb)

# Day 15 — Einstein Summation: The Notation Language of ML

**Category:** ML Engineering | **Format:** short-coding | **Difficulty:** standard (~25 min)

**Relevance:** `np.einsum` (and its PyTorch equivalent) is the notation that transformer code lives in. Every ARENA exercise, every circuit analysis script, and every model implementation in SAIGE and LASR is written in einsum. The LASR CodeSignal Python ML module tests tensor ops fluency directly — and every circuit analysis tool built this month (logit lens, activation patching, DLA) is one einsum call at its core.

## Problem Summary

Implement three functions using `np.einsum` — covering the three canonical patterns that appear constantly in ML code:
1. `einsum_outer` — outer product of two 1D vectors
2. `einsum_batch_matmul` — batched matrix multiply
3. `einsum_attention` — single-head scaled dot-product attention

## The Idea

Einstein summation notation describes a tensor contraction with a single string. The convention:

- **Each letter** is a dimension name (e.g. `b` = batch, `i` = query position, `d` = head dimension)
- **A letter that appears in both inputs but not the output** is *contracted* — summed over
- **A letter in all three** (both inputs + output) is a *batch* dimension — the operation runs independently per slice
- **A letter in only one input and the output** is preserved as-is

Three patterns cover most of ML:

| Pattern | einsum string | What it computes |
|---------|--------------|------------------|
| Outer product | `'i,j->ij'` | every pair (i, j) multiplied |
| Batch matmul | `'bik,bkj->bij'` | matrix product for each batch element |
| Attention logits | `'id,jd->ij'` | query×key dot products, one per (query, key) pair |

## Setup

In [ ]:
import numpy as np
np.random.seed(42)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

print("Setup complete.")

## Task 1 — `einsum_outer`

Compute the **outer product** of two 1D arrays using `np.einsum`.

**Input:** `u` shape `(n,)`, `v` shape `(m,)`

**Output:** shape `(n, m)` where `result[i, j] = u[i] * v[j]`

**Hint:** when no index is shared between the two inputs, einsum computes every pair — no summation, just broadcasting. The output subscript lists both indices.

In [ ]:
def einsum_outer(u, v):
    """
    Args:
        u: shape (n,)
        v: shape (m,)
    Returns:
        outer product, shape (n, m)
    """
    raise NotImplementedError

### Tests — Task 1

In [ ]:
u = np.array([1., 2., 3.])
v = np.array([4., 5.])
result = einsum_outer(u, v)

assert result.shape == (3, 2), f"Expected (3, 2), got {result.shape}"
assert np.allclose(result, np.outer(u, v)), f"Values mismatch\n  got: {result}\n  expected: {np.outer(u, v)}"

print(f"✓ Task 1 passed — outer product {u.shape} × {v.shape} → {result.shape}")
print(result)

## Task 2 — `einsum_batch_matmul`

Compute a **batched matrix multiply** using `np.einsum`.

**Input:** `A` shape `(batch, M, K)`, `B` shape `(batch, K, N)`

**Output:** shape `(batch, M, N)` — for each batch element, `result[b] = A[b] @ B[b]`

**Hint:** the shared contracted index `k` appears in both `A` and `B` but not the output — einsum sums over it. The batch index `b` appears in all three, so it's preserved.

In [ ]:
def einsum_batch_matmul(A, B):
    """
    Args:
        A: shape (batch, M, K)
        B: shape (batch, K, N)
    Returns:
        shape (batch, M, N)
    """
    raise NotImplementedError

### Tests — Task 2

In [ ]:
A = np.random.randn(4, 3, 5)
B = np.random.randn(4, 5, 7)
result = einsum_batch_matmul(A, B)

assert result.shape == (4, 3, 7), f"Expected (4, 3, 7), got {result.shape}"

ref = np.stack([A[i] @ B[i] for i in range(4)])
assert np.allclose(result, ref, atol=1e-6), "Should match loop-based batch matmul"

print(f"✓ Task 2 passed — batch matmul {A.shape} × {B.shape} → {result.shape}")

## Task 3 — `einsum_attention`

Implement **single-head scaled dot-product attention** using only `np.einsum` and `softmax`.

**Input:** `Q`, `K`, `V` each shape `(T, d)` (T = sequence length, d = head dimension)

**Output:** `(output, attn_weights)` where `output` is shape `(T, d)` and `attn_weights` is shape `(T, T)`

**Three steps:**
1. Compute logits: each query position `i` dot-products with each key position `j` over the head dimension `d` — subscript `'id,jd->ij'`
2. Scale by `1/sqrt(d)` and apply softmax over the key axis (axis=-1) to get attention weights
3. Aggregate values: each query position `i` attends to all value positions `j` — subscript `'ij,jd->id'`

**Why scale?** Without `1/sqrt(d)`, dot products grow with d, pushing softmax into saturation (near-zero gradients). This is equation (1) in *Attention Is All You Need* (Vaswani et al., 2017).

In [ ]:
def einsum_attention(Q, K, V):
    """
    Args:
        Q: shape (T, d)
        K: shape (T, d)
        V: shape (T, d)
    Returns:
        output:       shape (T, d)
        attn_weights: shape (T, T) — rows sum to 1.0
    """
    raise NotImplementedError

### Tests — Task 3

In [ ]:
T, d = 6, 8
Q = np.random.randn(T, d)
K = np.random.randn(T, d)
V = np.random.randn(T, d)

output, weights = einsum_attention(Q, K, V)

assert output.shape == (T, d), f"Expected ({T}, {d}), got {output.shape}"
assert weights.shape == (T, T), f"Expected ({T}, {T}), got {weights.shape}"
assert np.allclose(weights.sum(axis=-1), 1.0, atol=1e-6), \
    f"Attention weights must sum to 1 per query, got {weights.sum(axis=-1)}"
assert np.all(weights >= 0), "Attention weights must be non-negative"

# Verify against explicit reference
logits_ref = (Q @ K.T) / np.sqrt(d)
weights_ref = softmax(logits_ref, axis=-1)
output_ref  = weights_ref @ V
assert np.allclose(output, output_ref, atol=1e-6), "Output doesn't match reference"
assert np.allclose(weights, weights_ref, atol=1e-6), "Weights don't match reference"

print(f"✓ Task 3 passed — output {output.shape}, weights {weights.shape}, row sums all ≈1.0")

## Reflection

These three einsum patterns — outer product, batch matmul, attention — compose into every transformer operation. Reading ARENA Chapter 2–4 exercises with this notation in mind makes the code nearly self-documenting: each subscript string is a precise statement of what information flows where. The attention pattern (`'id,jd->ij'` then `'ij,jd->id'`) is also a lens: it tells you that attention is fundamentally about *which positions borrow information from which other positions* over the head dimension.

📚 **Further reading:** Vaswani et al. (2017), *Attention Is All You Need* — Section 3.2.1 (Scaled Dot-Product Attention). After this challenge, the paper's equations map directly to your einsum strings. [arxiv.org/pdf/1706.03762](https://arxiv.org/pdf/1706.03762)

<details>
<summary><b>Solution</b></summary>

```python
def einsum_outer(u, v):
    return np.einsum('i,j->ij', u, v)

def einsum_batch_matmul(A, B):
    return np.einsum('bik,bkj->bij', A, B)

def einsum_attention(Q, K, V):
    d = Q.shape[-1]
    logits  = np.einsum('id,jd->ij', Q, K) / np.sqrt(d)  # (T, T)
    weights = softmax(logits, axis=-1)                     # (T, T)
    output  = np.einsum('ij,jd->id', weights, V)          # (T, d)
    return output, weights
```

**Key insight — Task 1:** In `'i,j->ij'`, no index is shared between the two inputs, so there's no contraction — einsum simply computes every (i, j) pair. This is the outer product.

**Key insight — Task 2:** In `'bik,bkj->bij'`, the index `k` appears in both inputs but not the output — einsum sums over it (contracts). The index `b` appears in all three, so it's preserved as a batch dimension.

**Key insight — Task 3:** Two complementary patterns:
- `'id,jd->ij'`: both `i` and `j` appear in the output (all position pairs preserved), `d` is contracted (dot product over head dim) — this is the attention logit matrix.
- `'ij,jd->id'`: `j` is contracted (each query position `i` aggregates over all key/value positions `j`), `d` is preserved — this is value aggregation.

**Why this matters for mech interp:** Every circuit analysis tool (logit lens, activation patching, DLA) is ultimately a set of einsum operations on the residual stream. Understanding the subscript language means you can read and write these tools without getting lost in shapes.
</details>